# WaferGuard FAB Anomaly Detection Performance Workbench

이 notebook은 runtime feature를 leakage 없이 비교하기 위한 성능 개선 실험용입니다. Synthetic metric은 실제 FAB 성능을 의미하지 않습니다. **기본 상태에서는 어떤 모델도 학습하지 않으며**, EDA, label audit, split preview, 비학습 feature 분석까지만 수행합니다.

## 1. Configuration

In [ ]:
from copy import deepcopy
from pathlib import Path
import json
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import JSON, display
from app.services.fab_analysis import (
    BASELINE_PREDICTION_COLUMN, DATA_INPUT_DIR, FAB_ANALYSIS_OUTPUT_DIR,
    GROUND_TRUTH_COLUMN, build_workbench_dashboard_summary, correlation_table,
    export_runtime_features, load_local_dataset, save_workbench_dashboard_summary,
)
from app.services.fab_experiment import (
    CANDIDATE_SEARCH_SPACE, audit_label_sources, baseline_metrics,
    build_candidate_artifact, build_feature_sets, candidate_grid, context_false_positives,
    detection_delays, error_examples, evaluate_locked_test, experiment_manifest, file_sha256,
    grouped_chronological_split,
    lock_validation_winner, non_training_feature_analysis, per_fault_metrics,
    run_validation_experiment, save_candidate_artifact, save_json_output, save_table_output,
    seed_robustness, validate_split_integrity, warn_if_final_test_reused,
    workbench_dataset_profile,
)
from app.services.fab_schema import FAB_FEATURE_CONTRACT, config_fingerprint

PROCESS_ID = 'cmp'
DATA_FILE = DATA_INPUT_DIR / 'fab_training.csv'
EXPORT_FROM_LOCAL_DB = False
RUN_EXPERIMENT = False
RUN_FINAL_TEST = False
CREATE_CANDIDATE = False
RANDOM_STATE = 42
SPLIT_RATIOS = (0.60, 0.20, 0.20)
CANDIDATE_GRID = deepcopy(CANDIDATE_SEARCH_SPACE)
print({'RUN_EXPERIMENT': RUN_EXPERIMENT, 'RUN_FINAL_TEST': RUN_FINAL_TEST, 'CREATE_CANDIDATE': CREATE_CANDIDATE})

## 2. Data Load
CSV/Parquet 또는 명시적으로 요청한 local runtime DB export를 사용합니다.

In [ ]:
if EXPORT_FROM_LOCAL_DB:
    DATA_FILE = export_runtime_features(PROCESS_ID, DATA_INPUT_DIR / f'{PROCESS_ID}_runtime_features.csv')
df = load_local_dataset(DATA_FILE)
audit_label_sources(df, include_counts=False)  # provenance validation only
splits = grouped_chronological_split(df, ratios=SPLIT_RATIOS, normal_only_train=True)
development = pd.concat([splits.train, splits.validation], ignore_index=True)
print(f'Loaded {len(df):,} rows x {len(df.columns):,} columns from {DATA_FILE}')
preview_columns = [name for name in development.columns if not str(name).startswith('ground_truth_')]
display(development[preview_columns].head())
print('Preview uses Train/Validation rows only; Test values and labels remain sealed.')

## 3. Data Quality

In [ ]:
profile = workbench_dataset_profile(df, label_audit_frame=development)
display(JSON(profile))
display(pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing': df.isna().sum(), 'unique': df.nunique(dropna=False)}))

## 4. Label Source Audit

`baseline_detector_is_anomaly`는 detector prediction이며 feature/label이 아닙니다. `ground_truth_*`는 `simulation_faults`에서 온 평가 전용 값입니다.

In [ ]:
label_audit = audit_label_sources(development)
display(JSON(label_audit))
if not label_audit['ground_truth_available']:
    print('supervised performance evaluation unavailable; EDA/unsupervised configuration only')

## 5. Data Distribution / EDA

In [ ]:
runtime_prefixes = ('z:', 'delta:', 'rel:', 'roll_mean_delta:', 'roll_std:', 'context:')
runtime_features = [name for name in df.columns if str(name).startswith(runtime_prefixes)]
FEATURE_SETS = build_feature_sets(runtime_features)
display(pd.DataFrame([{'feature_set': name, 'count': len(columns), 'features': columns} for name, columns in FEATURE_SETS.items()]))
if GROUND_TRUTH_COLUMN in development:
    display(development[GROUND_TRUTH_COLUMN].value_counts(dropna=False).rename('rows').to_frame())
if 'ground_truth_fault_type' in development:
    display(development['ground_truth_fault_type'].value_counts(dropna=False).rename('rows').to_frame())
print('Test labels and fault distribution remain sealed until RUN_FINAL_TEST=True.')

## 6. Split Preview
동일 wafer/process run은 한 split에만 속하며 시간 순서를 보존합니다. Train은 GT가 있으면 normal-only입니다.

In [ ]:
validate_split_integrity(splits.train, splits.validation, splits.test)
display(splits.manifest)
display(pd.DataFrame([
    {'split': 'train', 'rows': len(splits.train), 'runs': splits.train.process_run_id.nunique()},
    {'split': 'validation', 'rows': len(splits.validation), 'runs': splits.validation.process_run_id.nunique()},
    {'split': 'test (sealed)', 'rows': len(splits.test), 'runs': splits.test.process_run_id.nunique()},
]))
print('split_hash:', splits.split_hash, 'normal_only_applied:', splits.normal_only_applied)

## 7. Baseline Review

In [ ]:
baseline_validation = baseline_metrics(splits.validation)
display(JSON(baseline_validation)) if baseline_validation else print('Baseline comparison unavailable')

## 8. Feature Analysis (no model fit)
Correlation과 standardized mean difference는 EDA용입니다. Feature set 선택은 Validation 비교로만 결정합니다.

In [ ]:
feature_analysis = non_training_feature_analysis(development, FEATURE_SETS)
display(feature_analysis['feature_group_summary'])
display(feature_analysis['feature_effects'].head(30))
display(feature_analysis['correlation'])

## 9. Candidate Configuration
기존 Robust Z, LedoitWolf Mahalanobis, Isolation Forest, One-Class SVM factory를 재사용합니다. 아직 fit하지 않습니다.

In [ ]:
candidate_plan = pd.DataFrame(candidate_grid(CANDIDATE_GRID))
display(candidate_plan)
print('candidate combinations x non-empty feature sets:', len(candidate_plan), 'x', sum(bool(v) for v in FEATURE_SETS.values()))

## 10. Validation Experiment
이 셀은 `RUN_EXPERIMENT=True`일 때만 Train fit 및 Validation threshold 선택을 수행합니다. Test는 보지 않습니다.

In [ ]:
experiment_result = None
locked_candidate = None
if RUN_EXPERIMENT:
    experiment_result = run_validation_experiment(
        splits, FEATURE_SETS, search_space=CANDIDATE_GRID, random_state=RANDOM_STATE
    )
    leaderboard = experiment_result['leaderboard']
    display(leaderboard)
    winner = leaderboard.iloc[0].to_dict()
    winner_features = FEATURE_SETS[winner['feature_set']]
    feature_contract = str(df['feature_contract'].dropna().iloc[0]) if 'feature_contract' in df and df['feature_contract'].notna().any() else FAB_FEATURE_CONTRACT
    config_fp = str(df['config_fingerprint'].dropna().iloc[0]) if 'config_fingerprint' in df and df['config_fingerprint'].notna().any() else 'unavailable'
    locked_candidate = lock_validation_winner(
        winner, feature_names=winner_features, feature_contract=feature_contract,
        config_fingerprint_value=config_fp, split_hash=splits.split_hash,
    )
    display(JSON(locked_candidate))
else:
    print('Skipped: set RUN_EXPERIMENT=True yourself to fit candidates.')

## 11. Error Analysis

In [ ]:
fault_metrics_table = pd.DataFrame()
context_fp_table = pd.DataFrame()
error_examples_table = pd.DataFrame()
seed_analysis = {'status': 'not_run'}
if experiment_result is not None:
    winner_id = experiment_result['leaderboard'].iloc[0]['candidate_id']
    winner_model = experiment_result['fitted_candidates'][winner_id]
    winner_features = locked_candidate['feature_names']
    validation_scores = -winner_model.decision_function(splits.validation[winner_features])
    validation_predicted = validation_scores >= locked_candidate['threshold']
    fault_metrics_table = per_fault_metrics(splits.validation, validation_predicted)
    context_fp_table = context_false_positives(splits.validation, validation_predicted)
    error_examples_table = error_examples(splits.validation, validation_predicted)
    seed_analysis = seed_robustness(splits.validation, validation_predicted)
    display(fault_metrics_table)
    display(context_fp_table)
    display(error_examples_table)
    display(detection_delays(splits.validation, validation_predicted))
    display(JSON(seed_analysis))
else:
    print('Error analysis waits for an opt-in Validation winner.')

## 12. Final Test
`RUN_FINAL_TEST=True`일 때만 locked feature/model/hyperparameter/threshold 그대로 Test를 한 번 평가합니다. 재튜닝하지 않습니다.

In [ ]:
final_test_metrics = None
baseline_test = None
if RUN_FINAL_TEST:
    if experiment_result is None or locked_candidate is None:
        raise RuntimeError('Run and lock a Validation experiment before Final Test.')
    winner_model = experiment_result['fitted_candidates'][experiment_result['leaderboard'].iloc[0]['candidate_id']]
    existing_final_path = FAB_ANALYSIS_OUTPUT_DIR / 'final_test_metrics.json'
    existing_final = json.loads(existing_final_path.read_text(encoding='utf-8')) if existing_final_path.is_file() else None
    warn_if_final_test_reused(locked_candidate, existing_final)
    final_test_metrics = evaluate_locked_test(winner_model, splits.test, locked_candidate)
    baseline_test = baseline_metrics(splits.test)
    if baseline_test:
        baseline_test = {**baseline_test, 'evaluation_split': 'test'}
    display(JSON({'candidate': final_test_metrics, 'production_baseline': baseline_test}))
else:
    print('Final Test remains sealed (RUN_FINAL_TEST=False).')

## 13. Candidate Artifact
자동 등록/승격은 하지 않습니다. Test dataset으로 재학습하지 않습니다.

In [ ]:
candidate_path = None
if CREATE_CANDIDATE:
    if experiment_result is None or locked_candidate is None:
        raise RuntimeError('Lock a Validation winner before candidate creation.')
    winner_model = experiment_result['fitted_candidates'][experiment_result['leaderboard'].iloc[0]['candidate_id']]
    bundle = build_candidate_artifact(
        winner_model, locked_candidate, process_id=PROCESS_ID,
        training_metadata={'train_rows': len(splits.train), 'validation_rows': len(splits.validation)},
    )
    candidate_path = save_candidate_artifact(
        bundle, FAB_ANALYSIS_OUTPUT_DIR / f'{PROCESS_ID}-locked-candidate.joblib', create_candidate=True
    )
    print('Candidate artifact:', candidate_path, '(not registered)')
else:
    print('Candidate creation disabled (CREATE_CANDIDATE=False).')

## 14. Summary

In [ ]:
feature_contract_value = str(df['feature_contract'].dropna().iloc[0]) if 'feature_contract' in df and df['feature_contract'].notna().any() else FAB_FEATURE_CONTRACT
config_fp_value = str(df['config_fingerprint'].dropna().iloc[0]) if 'config_fingerprint' in df and df['config_fingerprint'].notna().any() else 'unavailable'
feature_fp_value = config_fingerprint({'feature_contract': feature_contract_value, 'feature_names': FEATURE_SETS['all_features']})
manifest = experiment_manifest(
    process_id=PROCESS_ID, source_file=DATA_FILE, source_hash=file_sha256(DATA_FILE),
    feature_contract=feature_contract_value, feature_fingerprint=feature_fp_value,
    config_fingerprint_value=config_fp_value, random_state=RANDOM_STATE,
    split_definition={'ratios': SPLIT_RATIOS, 'split_hash': splits.split_hash},
    candidate_grid_value=candidate_grid(CANDIDATE_GRID),
)
outputs = {
    'dataset_profile': str(save_json_output('dataset_profile.json', profile, output_dir=FAB_ANALYSIS_OUTPUT_DIR)),
    'split_manifest': str(save_table_output('split_manifest.csv', splits.manifest, output_dir=FAB_ANALYSIS_OUTPUT_DIR)),
    'feature_importance': str(save_table_output('feature_importance.csv', feature_analysis['feature_effects'], output_dir=FAB_ANALYSIS_OUTPUT_DIR)),
    'correlation_matrix': str(save_table_output('correlation_matrix.csv', feature_analysis['correlation'], output_dir=FAB_ANALYSIS_OUTPUT_DIR)),
    'experiment_manifest': str(save_json_output('experiment_manifest.json', manifest, output_dir=FAB_ANALYSIS_OUTPUT_DIR)),
}
if experiment_result is not None:
    outputs['candidate_leaderboard'] = str(save_table_output('candidate_leaderboard.csv', experiment_result['leaderboard'], output_dir=FAB_ANALYSIS_OUTPUT_DIR))
    outputs['validation_metrics'] = str(save_table_output('validation_metrics.csv', experiment_result['leaderboard'], output_dir=FAB_ANALYSIS_OUTPUT_DIR))
    outputs['locked_candidate'] = str(save_json_output('locked_candidate.json', locked_candidate, output_dir=FAB_ANALYSIS_OUTPUT_DIR))
    outputs['per_fault_metrics'] = str(save_table_output('per_fault_metrics.csv', fault_metrics_table, output_dir=FAB_ANALYSIS_OUTPUT_DIR))
    outputs['context_false_positives'] = str(save_table_output('context_false_positives.csv', context_fp_table, output_dir=FAB_ANALYSIS_OUTPUT_DIR))
if final_test_metrics is not None:
    outputs['final_test_metrics'] = str(save_json_output('final_test_metrics.json', final_test_metrics, output_dir=FAB_ANALYSIS_OUTPUT_DIR))
if candidate_path is not None:
    outputs['candidate_artifact'] = str(candidate_path)
_, top_correlations = correlation_table(development[FEATURE_SETS['all_features']])
summary = build_workbench_dashboard_summary(
    process_id=PROCESS_ID, dataset=profile, baseline_metrics=baseline_test or baseline_validation,
    validation_winner=locked_candidate, final_test_metrics=final_test_metrics,
    per_fault_metrics=fault_metrics_table.to_dict('records'),
)
summary['feature_importance'] = feature_analysis['feature_effects'].head(20).to_dict('records')
summary['top_correlations'] = top_correlations
summary['seed_robustness'] = seed_analysis
summary['artifacts'] = outputs
outputs['dashboard_summary'] = str(save_workbench_dashboard_summary(summary, FAB_ANALYSIS_OUTPUT_DIR / 'dashboard_summary.json'))
display(JSON({**summary, 'artifacts': outputs}))